In [0]:
offline_students_schema = "ID string, FirstName string, LastName string, Address string, Skills string, Contacts string"
offline_students_raw_df = (
    spark.read
    .format("csv") 
    .option("header", "true")
    .schema(schema=offline_students_schema)
    .load(path= "/Volumes/dev/spark_db/datasets/spark_programming/data/students_offline.csv")
)
offline_students_raw_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students_raw")

In [0]:
%sql

select * from dev.spark_db.offline_students_raw

In [0]:
%sql 

 select id, from_json(address,
      """struct<AddressLine1 string,
        AddressLine2 string,
        City string,
        Country string,
        Pin string,
        State string>
      """) as address
from dev.spark_db.offline_students_raw

In [0]:
from pyspark.sql.functions import from_json

address_schema = "struct<AddressLine1 string, AddressLine2 string, City string, Country string, Pin string, State string>"
skills_schema = "array<struct<Skill string, YearsOfExperience string>>"
contacts_schema = "map<string, string>"

offline_students_df = (
    offline_students_raw_df.withColumns({
        "address": from_json("address", address_schema),
        "skills": from_json("skills", skills_schema),
        "contacts": from_json("contacts", contacts_schema)
    })
)

#offline_students_df.display()
offline_students_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students")

In [0]:
%sql

select * from dev.spark_db.offline_students

####4. Requirement
Perform the following analysis
1. What is country wise student count.
2. Find all students with more than 1 years of Spark knowledge
3. Find all students who didn't provide phone or whatsapp

Find all students with more than 1 years of Spark knowledge

In [0]:
%sql

select students
from dev.spark_db.offline_students
where skills.skill like "%Spark%"
and skills.YearsOfExperience >1

1. What is country wise student count.

In [0]:
%sql
select address.Country, count(*) as count
from dev.spark_db.offline_students
group by address.Country